In [1]:
import numpy as np

# =============================================================================
# Berry Phase Truncation Error Bound
#
# Bound: |Δγ_N| ≤ Σ_j arctan( ε_j·ε_{j+1} / ((1-ε_j)(1-ε_{j+1})|O_D(j)| )
#
# Phase model from Fig. 4B (slope ±1/2 reading):
#   arg(β/α) = (θ - π)/2  →  a_11(θ) = √p_11 · e^{i(θ-π)/2}
#   arg(γ/α) = (π - θ)/2  →  a_20(θ) = √p_20 · e^{i(π-θ)/2}
#   α taken as real gauge:  a_02(θ) = √p_02
# =============================================================================

# -----------------------------------------------------------------------------
# Probability vectors from hardware (Hanoi, Apr 4)
# Format: (angle θ, probability vector)
# Basis ordering: |00>,|01>,|02>,|10>,|11>,|12>,|20>,|21>,|22>
# Dominant indices: 2 (|02>), 4 (|11>), 6 (|20>)
# -----------------------------------------------------------------------------

w = {}
w[0]  = (0,              np.array([ 0.01162158, -0.00465488,  0.33739193,  0.0072317,
                                     0.31953348,  0.00605047,  0.30543009,  0.02578479, -0.00838916]))
w[1]  = (2*np.pi/3,      np.array([ 0.00067024, -0.00236954,  0.43210499, -0.00177051,
                                     0.10078978,  0.01457336,  0.39321297,  0.03774422,  0.02504449]))
w[2]  = (np.pi/2,        np.array([ 2.12115478e-02,  3.46200781e-03,  3.47218970e-01,  1.57590447e-02,
                                     1.84381092e-01,  1.61194434e-04,  3.57202571e-01,  2.98836136e-02,  4.07199590e-02]))
w[3]  = (2*np.pi/5,      np.array([ 0.03142523,  0.00194623,  0.35768344,  0.00328126,
                                     0.21747801,  0.01430082,  0.32693466,  0.02654251,  0.02040784]))
w[4]  = (np.pi/3,        np.array([ 2.21261481e-02, -1.36740039e-04,  3.20134642e-01,  4.79971741e-03,
                                     2.67555581e-01,  4.67825507e-03,  3.39997716e-01,  2.26352755e-02,  1.82094055e-02]))
w[5]  = (2*np.pi/7,      np.array([ 0.01091598, -0.00241662,  0.35073897,  0.01036843,
                                     0.29054482, -0.00758829,  0.29930806,  0.03246427,  0.01566437]))
w[6]  = (4*np.pi/3,      np.array([ 0.01443338, -0.00759534,  0.39960547,  0.0082209,
                                     0.09052253, -0.00192335,  0.39951553,  0.03615589,  0.06106499]))
w[7]  = (4*np.pi/5,      np.array([ 6.80718910e-03,  6.02283556e-03,  4.43831627e-01, -3.66688786e-03,
                                     6.38676080e-02,  2.86909548e-02,  4.20161469e-01,  3.43499846e-02, -6.47802471e-05]))
w[8]  = (4*np.pi/7,      np.array([ 0.02959617,  0.00259343,  0.36817979,  0.00788336,
                                     0.18920559,  0.00494269,  0.39323909,  0.02657886, -0.022219]))
w[9]  = (3*np.pi/2,      np.array([ 0.00722698,  0.00380351,  0.39609568, -0.00232187,
                                     0.19588625,  0.02059617,  0.34785905,  0.02012307,  0.01073116]))
w[10] = (6*np.pi/5,      np.array([ 0.01687536,  0.00418956,  0.4172253,  -0.00176515,
                                     0.04644531,  0.02833333,  0.40482875,  0.03107608,  0.05279148]))
w[11] = (6*np.pi/7,      np.array([ 0.02584046,  0.00547339,  0.41774764, -0.00507892,
                                     0.02374591,  0.0311637,   0.42527601,  0.02961148,  0.04622033]))
w[12] = (5*np.pi/3,      np.array([ 0.005196,    0.00157121,  0.37354749,  0.002142,
                                     0.2548247,   0.01558938,  0.31573322,  0.02752265,  0.00387336]))
w[13] = (8*np.pi/7,      np.array([ 0.02349984, -0.0057321,   0.43988562, -0.00135873,
                                     0.04005334,  0.0423582,   0.44576959,  0.03916698, -0.02364274]))
w[14] = (10*np.pi/7,     np.array([ 0.01175623,  0.00166792,  0.37777297,  0.0122775,
                                     0.16938342,  0.00785588,  0.39387426,  0.03347079, -0.00805898]))
w[15] = (12*np.pi/7,     np.array([ 0.01449329, -0.00143568,  0.34005056,  0.00144727,
                                     0.23476494,  0.01983087,  0.33024341,  0.03142149,  0.02918384]))
w[16] = (np.pi,          np.array([-0.01654823,  0.01369785,  0.45452254,  0.0087674,
                                    -0.00122485,  0.02452292,  0.47829132,  0.02643333,  0.01153772]))

# Dominant component indices: |02>=2, |11>=4, |20>=6
DOM = [2, 4, 6]

# -----------------------------------------------------------------------------
# Helper functions
# -----------------------------------------------------------------------------

def dominant_weight(prob):
    """Sum of dominant component probabilities."""
    return sum(max(prob[i], 0) for i in DOM)

def epsilon(prob):
    """Non-dominant population weight ε = 1 - Σ p_dominant."""
    return 1.0 - dominant_weight(prob)

def complex_overlap(theta_j, prob_j, theta_k, prob_k):
    """
    Complex overlap <d_j|d_k> between normalized dominant states.

    Phase model (Fig. 4B, slope ±1/2):
        a_02 = √p_02            (real, gauge choice)
        a_11 = √p_11 · e^{i(θ-π)/2}
        a_20 = √p_20 · e^{i(π-θ)/2}

    <d_j|d_k> = [A_02 + A_11·e^{iΔθ/2} + A_20·e^{-iΔθ/2}] / √(dom_j·dom_k)
    where Δθ = θ_k - θ_j
    """
    p02j = max(prob_j[2], 0); p11j = max(prob_j[4], 0); p20j = max(prob_j[6], 0)
    p02k = max(prob_k[2], 0); p11k = max(prob_k[4], 0); p20k = max(prob_k[6], 0)
    dj = dominant_weight(prob_j)
    dk = dominant_weight(prob_k)

    A02 = np.sqrt(p02j * p02k)
    A11 = np.sqrt(p11j * p11k)
    A20 = np.sqrt(p20j * p20k)
    dtheta = (theta_k - theta_j) / 2.0  # slope-1/2 per Fig. 4B

    z = A02 + A11 * np.exp(1j * dtheta) + A20 * np.exp(-1j * dtheta)
    return z / np.sqrt(dj * dk)

def per_step_bound(theta_j, prob_j, theta_k, prob_k):
    """Per-step truncation error bound at step (j, j+1)."""
    ov   = complex_overlap(theta_j, prob_j, theta_k, prob_k)
    OD   = abs(ov)**2          # |<d_j|d_{j+1}>|^2 = Tr(rho_D(j) rho_D(j+1))
    ej   = epsilon(prob_j)
    ek   = epsilon(prob_k)
    return np.arctan(ej * ek / ((1 - ej) * (1 - ek) * OD))

# -----------------------------------------------------------------------------
# Partition configurations (from notebook)
# Each entry: (w_index, angle_override or None)
# angle_override needed when same probability vector is reused at a different θ
# (exploiting symmetry p(θ) = p(2π-θ) for AKLT ground states)
# -----------------------------------------------------------------------------

configs = {
    'N=3': [(0, None), (1,  None), (6,  None), (0,  2*np.pi)],
    'N=4': [(0, None), (2,  None), (16, None), (9,  None),   (0,  2*np.pi)],
    'N=5': [(0, None), (3,  None), (7,  None), (10, None),   (3,  8*np.pi/5), (0, 2*np.pi)],
    'N=6': [(0, None), (4,  None), (1,  None), (16, None),   (6,  None), (12, None), (0, 2*np.pi)],
    'N=7': [(0, None), (5,  None), (8,  None), (11, None),   (13, None), (14, None), (15, None), (0, 2*np.pi)],
}

# -----------------------------------------------------------------------------
# Compute bounds
# -----------------------------------------------------------------------------

print("=" * 80)
print("BERRY PHASE TRUNCATION ERROR BOUND")
print("  |Δγ_N| ≤ Σ_j arctan( ε_j·ε_{j+1} / ((1-ε_j)(1-ε_{j+1})·|O_D(j)|) )")
print("=" * 80)

summary = {}

for N_label, seq in configs.items():
    print(f"\n{N_label}  (code N = {len(seq)}, overlaps = {len(seq)-1})")
    print(f"  {'Step':>4}  {'θ_j/π':>7}  {'θ_{j+1}/π':>9}  {'ε_j':>7}  {'ε_{j+1}':>8}  "
          f"{'|O_D|':>7}  {'per-step bound':>14}")
    print("  " + "-" * 70)

    total_bound = 0.0
    O_min       = np.inf
    step_data   = []

    for i in range(len(seq) - 1):
        idx_j, ang_j = seq[i]
        idx_k, ang_k = seq[i + 1]
        theta_j = ang_j if ang_j is not None else w[idx_j][0]
        theta_k = ang_k if ang_k is not None else w[idx_k][0]
        prob_j  = w[idx_j][1]
        prob_k  = w[idx_k][1]

        ov      = complex_overlap(theta_j, prob_j, theta_k, prob_k)
        OD      = abs(ov)**2
        ej      = epsilon(prob_j)
        ek      = epsilon(prob_k)
        ps      = per_step_bound(theta_j, prob_j, theta_k, prob_k)

        total_bound += ps
        O_min        = min(O_min, OD)
        step_data.append((theta_j, theta_k, ej, ek, OD, ps))

        print(f"  {i+1:>4}  {theta_j/np.pi:>7.4f}  {theta_k/np.pi:>9.4f}  "
              f"{ej:>7.4f}  {ek:>8.4f}  {OD:>7.4f}  {ps:>12.6f} rad")

    summary[N_label] = (total_bound, O_min)
    print(f"  {'':>4}  {'':>7}  {'':>9}  {'':>7}  {'':>8}  "
          f"O_min={O_min:.4f}  TOTAL = {total_bound:.4f} rad  ({100*total_bound/np.pi:.1f}% of π)")

# -----------------------------------------------------------------------------
# Summary table
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("SUMMARY")
print(f"  {'N':>4}  {'Bound (rad)':>12}  {'% of π':>8}")
print("  " + "-" * 28)
worst = 0.0
for N_label, (total, O_min) in summary.items():
    print(f"  {N_label:>4}  {total:>12.4f}  {100*total/np.pi:>7.1f}%")
    worst = max(worst, total)

print(f"\n  Worst-case bound across all N: {worst:.4f} rad ({100*worst/np.pi:.1f}% of π)")
print("=" * 80)


BERRY PHASE TRUNCATION ERROR BOUND
  |Δγ_N| ≤ Σ_j arctan( ε_j·ε_{j+1} / ((1-ε_j)(1-ε_{j+1})·|O_D(j)|) )

N=3  (code N = 4, overlaps = 3)
  Step    θ_j/π  θ_{j+1}/π      ε_j   ε_{j+1}    |O_D|  per-step bound
  ----------------------------------------------------------------------
     1   0.0000     0.6667   0.0376    0.0739   0.4900      0.006369 rad
     2   0.6667     1.3333   0.0739    0.1104   0.6134      0.016133 rad
     3   1.3333     2.0000   0.1104    0.0376   0.4872      0.009960 rad
                                               O_min=0.4872  TOTAL = 0.0325 rad  (1.0% of π)

N=4  (code N = 5, overlaps = 4)
  Step    θ_j/π  θ_{j+1}/π      ε_j   ε_{j+1}    |O_D|  per-step bound
  ----------------------------------------------------------------------
     1   0.0000     0.5000   0.0376    0.1112   0.6577      0.007441 rad
     2   0.5000     1.0000   0.1112    0.0672   0.6765      0.013319 rad
     3   1.0000     1.5000   0.0672    0.0602   0.6743      0.006837 rad
     4   1.

In [2]:
# =============================================================================
# TRUNCATION BOUND, GAUGE-HONEST REFINEMENT
#
# (1) The neglected D-R cross term
#         Tr(rho_DR^j rho_RD^k) + Tr(rho_RD^j rho_DR^k)
#     is a number plus its own conjugate, hence REAL.
#
# (2) The Berry phase takes Arg(.).  For a real perturbation w on a complex
#     overlap z:   Delta Arg  ~  - w sin(Arg z)/|z|     (NOT w/|z|)
#     So each step carries a factor |sin(phi_j)|, phi_j = per-step increment.
#
# (3) GAUGE.  A gauge change at angle j shifts phi_{j-1} by +chi_j and phi_j by
#     -chi_j: it redistributes phase between steps but leaves sum_j phi_j = gamma
#     invariant.  So phi_j is NOT gauge invariant, but its sum is.
#     Since sin is CONCAVE on [0,pi], sum_j sin(phi_j) subject to fixed
#     sum_j phi_j is MAXIMISED at uniform phi_j = gamma/N.  The bound is
#     proportional to that sum, so uniform increments give the LARGEST bound and
#     therefore the SMALLEST delta*.  phi_j = pi/N is thus the worst case over
#     all gauges, not an assumption about ours.
#     (Valid only if all phi_j lie in [0,pi]; checked below.)
# =============================================================================
import numpy as np

TARGET_SCATTER = 0.31          # rad, observed scatter across partitions, Fig. 6B
GAMMA          = np.pi         # total Berry phase
grid           = np.linspace(0, 0.8, 8001)

# -----------------------------------------------------------------------------
# OPTIONAL: supply the TRUE per-step phases from the routine that actually
# produces gamma = pi in Fig. 6B.  Format {'N=3': [phi_1, phi_2, phi_3], ...}
# Leave as None to use only the gauge-independent bounds.
# -----------------------------------------------------------------------------
TRUE_PHASES = None

def step_terms(seq):
    """Return per-step (eps_j, eps_k, |O_D|) for a partition."""
    out = []
    for i in range(len(seq) - 1):
        (ij, aj), (ik, ak) = seq[i], seq[i + 1]
        tj = aj if aj is not None else w[ij][0]
        tk = ak if ak is not None else w[ik][0]
        pj, pk = w[ij][1], w[ik][1]
        z  = complex_overlap(tj, pj, tk, pk)
        out.append((epsilon(pj), epsilon(pk), abs(z)**2, np.angle(z)))
    return out

def bound(seq, delta, sinfac):
    """sinfac: None -> 1 (no refinement); float -> uniform; list -> per-step."""
    terms = step_terms(seq)
    tot = 0.0
    for i, (ej, ek, OD, _) in enumerate(terms):
        if sinfac is None:      f = 1.0
        elif np.isscalar(sinfac): f = sinfac
        else:                   f = abs(np.sin(sinfac[i]))
        tot += np.arctan(f * (ej*ek + 2*delta**2) / ((1-ej)*(1-ek)*OD))
    return tot

def dstar(seq, sinfac):
    v = np.array([bound(seq, d, sinfac) for d in grid])
    return grid[np.argmax(v > TARGET_SCATTER)] if (v > TARGET_SCATTER).any() else np.nan

# --- DIAGNOSTIC: does the reconstructed overlap carry the right total phase? ---
print("=" * 78)
print("DIAGNOSTIC: sum of reconstructed per-step Arg")
print("  Gauge freedom REDISTRIBUTES phase between steps but cannot change the")
print("  total.  A total != pi therefore indicates a reconstruction problem,")
print("  not a gauge choice.  Most likely cause: amplitudes built as sqrt(p) are")
print("  positive by construction, losing the negative singlet amplitude that")
print("  makes gamma topological.")
print("=" * 78)
for lbl, seq in configs.items():
    phis = [t[3] for t in step_terms(seq)]
    s = sum(phis)
    inrange = all(0 <= p <= np.pi for p in phis) or all(-np.pi <= p <= 0 for p in phis)
    print(f"  {lbl}: sum Arg = {s:+.4f} rad = {s/np.pi:+.3f} pi   "
          f"same-signed & in range: {inrange}")

# --- delta*: three levels of assumption ---------------------------------------
print("\n" + "=" * 78)
print(f"{'N':>5} {'|dg|(0)':>9} | {'(a) no sin':>11} {'(b) uniform pi/N':>17} {'(c) actual phases':>18}")
print(f"{'':>5} {'':>9} | {'assumption-':>11} {'worst case over':>17} {'gauge-honest':>18}")
print(f"{'':>5} {'':>9} | {'free floor':>11} {'all gauges':>17} {'(needs TRUE_PHASES)':>18}")
print("=" * 78)
for lbl, seq in configs.items():
    N  = len(seq) - 1
    b0 = bound(seq, 0.0, None)
    da = dstar(seq, None)
    db = dstar(seq, np.sin(GAMMA / N))
    dc = dstar(seq, TRUE_PHASES[lbl]) if TRUE_PHASES else np.nan
    print(f"{lbl:>5} {b0:>9.4f} | {da:>11.3f} {db:>17.3f} {dc:>18.3f}")

# --- bound evaluated at the measured delta ------------------------------------
DELTA_EST, DELTA_UL = 0.050, 0.153      # 17-angle harmonic fit: central, 2-sigma UL
print("\n" + "=" * 78)
print(f"Bound at delta = {DELTA_EST} (estimate) and {DELTA_UL} (2-sigma UL),")
print("using the worst-case-over-gauges factor sin(pi/N)")
print("=" * 78)
for lbl, seq in configs.items():
    N = len(seq) - 1
    f = np.sin(GAMMA / N)
    be = bound(seq, DELTA_EST, f)
    bu = bound(seq, DELTA_UL,  f)
    print(f"  {lbl:>4}: est {be:.4f} rad ({100*be/np.pi:>4.1f}% of pi)"
          f"   2sig {bu:.4f} rad ({100*bu/np.pi:>4.1f}% of pi)")

worst_b = max(dstar(seq, np.sin(GAMMA/(len(seq)-1))) for seq in configs.values())
best_b  = min(dstar(seq, np.sin(GAMMA/(len(seq)-1))) for seq in configs.values())
print(f"\n  Binding delta* (worst N) = {best_b:.3f}")
print(f"  Measured delta = {DELTA_EST} (2-sigma UL {DELTA_UL}) -> "
      f"{'CLEARS' if DELTA_UL < best_b else 'DOES NOT CLEAR'} the crossover")

DIAGNOSTIC: sum of reconstructed per-step Arg
  Gauge freedom REDISTRIBUTES phase between steps but cannot change the
  total.  A total != pi therefore indicates a reconstruction problem,
  not a gauge choice.  Most likely cause: amplitudes built as sqrt(p) are
  positive by construction, losing the negative singlet amplitude that
  makes gamma topological.
  N=3: sum Arg = -0.8387 rad = -0.267 pi   same-signed & in range: True
  N=4: sum Arg = -0.9373 rad = -0.298 pi   same-signed & in range: True
  N=5: sum Arg = -0.7156 rad = -0.228 pi   same-signed & in range: True
  N=6: sum Arg = -0.7990 rad = -0.254 pi   same-signed & in range: True
  N=7: sum Arg = -0.7061 rad = -0.225 pi   same-signed & in range: False

    N   |dg|(0) |  (a) no sin  (b) uniform pi/N  (c) actual phases
                | assumption-   worst case over       gauge-honest
                |  free floor        all gauges (needs TRUE_PHASES)
  N=3    0.0325 |       0.145             0.157                nan
  N=4    

In [3]:
# =============================================================================
# SELF-VERIFICATION OF THE CLAIMS
#   (A) the neglected D-R cross term is REAL
#   (B) a REAL perturbation w on a complex overlap z shifts Arg(z) by
#           w * sin(Arg z) / |z|        NOT   w / |z|
#   (C) robustness of the 17-angle harmonic fit to dropping individual angles
# =============================================================================
import numpy as np
rng = np.random.default_rng(0)

# --- (A) cross term is real: Tr(A B^dag) + Tr(B A^dag) for random complex A,B -
print("=" * 72)
print("(A) cross term  Tr(rho_DR^j rho_RD^k) + Tr(rho_RD^j rho_DR^k)  is REAL")
print("    (rho_RD = rho_DR^dag, so the two terms are complex conjugates)")
print("=" * 72)
for trial in range(3):
    A = rng.normal(size=(3, 6)) + 1j*rng.normal(size=(3, 6))   # rho_DR at angle j
    B = rng.normal(size=(3, 6)) + 1j*rng.normal(size=(3, 6))   # rho_DR at angle k
    cross = np.trace(A @ B.conj().T) + np.trace(B @ A.conj().T)
    print(f"  trial {trial}:  cross = {cross.real:+.6f} {cross.imag:+.3e}i "
          f"  |Im|/|Re| = {abs(cross.imag)/abs(cross.real):.2e}")

# --- (B) numerical check of the phase-shift formula ---------------------------
print("\n" + "=" * 72)
print("(B) exact Arg(z+w) - Arg(z) vs the two candidate formulas, w REAL")
print(f"{'phi (rad)':>10} {'exact shift':>13} {'w*sin(phi)/|z|':>16} {'w/|z| (Eq.12)':>15} {'ratio':>8}")
print("=" * 72)
w_pert = 0.01
for phi in [np.pi/7, np.pi/5, np.pi/4, np.pi/3, np.pi/2, 2*np.pi/3]:
    z      = np.exp(1j*phi)                      # |z| = 1
    exact  = np.angle(z + w_pert) - np.angle(z)
    with_s = -w_pert*np.sin(phi)/abs(z)
    no_s   = -w_pert/abs(z)
    print(f"{phi:>10.4f} {exact:>13.6f} {with_s:>16.6f} {no_s:>15.6f} "
          f"{abs(no_s/exact):>8.2f}x")
print("  -> the sin(phi) form tracks the exact shift; Eq.(12) overestimates by")
print("     1/sin(phi), which at phi = pi/7 is a factor of 2.3")

# --- gauge/concavity check ----------------------------------------------------
print("\n" + "=" * 72)
print("(B2) concavity: sum_j sin(phi_j) at fixed sum_j phi_j = pi is MAXIMISED")
print("     by uniform phi_j -> uniform is the WORST CASE for the bound")
print("=" * 72)
for N in [3, 5, 7]:
    uni = N*np.sin(np.pi/N)
    worst = 0.0
    for _ in range(20000):                       # random gauges: random partitions of pi
        x = rng.dirichlet(np.ones(N))*np.pi
        worst = max(worst, np.sum(np.sin(x)))
    print(f"  N={N}: uniform = {uni:.4f}   max over 20000 random gauges = {worst:.4f}"
          f"   {'OK' if worst <= uni + 1e-3 else 'VIOLATED'}")

# --- (C) leave-one-out robustness of the harmonic fit -------------------------
print("\n" + "=" * 72)
print("(C) harmonic fit, leave-one-angle-out.  Needs w{} in scope.")
print("=" * 72)
try:
    DOMI = [2, 4, 6]
    ks   = sorted(w)
    th   = np.array([w[k][0] for k in ks])
    eps  = np.array([1 - sum(max(w[k][1][i], 0) for i in DOMI) for k in ks])
    p2s  = np.array([w[k][1][2] + w[k][1][6] for k in ks])
    p11  = np.array([w[k][1][4] for k in ks])

    def fit(mask):
        A = np.vstack([p2s[mask], p11[mask]]).T
        c, *_ = np.linalg.lstsq(A, eps[mask], rcond=None)
        r = eps[mask] - A @ c
        t = th[mask]
        X = np.vstack([np.ones_like(t), np.cos(t), np.sin(t)]).T
        cc, *_ = np.linalg.lstsq(X, r, rcond=None)
        dof = len(t) - 3
        s2  = np.sum((r - X @ cc)**2)/dof
        err = np.sqrt(s2*np.linalg.inv(X.T @ X)[1, 1])
        B   = np.hypot(cc[1], cc[2])
        return B, err

    B0, e0 = fit(np.ones(len(ks), bool))
    print(f"  all {len(ks)} angles:  B = {B0:.5f} +- {e0:.5f}   "
          f"2sig UL = {B0+2*e0:.5f}   delta <= {np.sqrt(B0+2*e0):.4f}")
    print(f"\n  {'dropped theta/pi':>17} {'B':>9} {'+-':>9} {'delta UL':>10}")
    for d in range(len(ks)):
        m = np.ones(len(ks), bool); m[d] = False
        B, e = fit(m)
        flag = "  <-- shifts B by >1 sigma" if abs(B-B0) > e0 else ""
        print(f"  {th[d]/np.pi:>17.3f} {B:>9.5f} {e:>9.5f} "
              f"{np.sqrt(B+2*e):>10.4f}{flag}")
except NameError:
    print("  (skipped: run this cell in the notebook where w{} is defined)")

(A) cross term  Tr(rho_DR^j rho_RD^k) + Tr(rho_RD^j rho_DR^k)  is REAL
    (rho_RD = rho_DR^dag, so the two terms are complex conjugates)
  trial 0:  cross = -7.354210 +0.000e+00i   |Im|/|Re| = 0.00e+00
  trial 1:  cross = -19.775493 +0.000e+00i   |Im|/|Re| = 0.00e+00
  trial 2:  cross = -1.979939 +0.000e+00i   |Im|/|Re| = 0.00e+00

(B) exact Arg(z+w) - Arg(z) vs the two candidate formulas, w REAL
 phi (rad)   exact shift   w*sin(phi)/|z|   w/|z| (Eq.12)    ratio
    0.4488     -0.004300        -0.004339       -0.010000     2.33x
    0.6283     -0.005831        -0.005878       -0.010000     1.72x
    0.7854     -0.007021        -0.007071       -0.010000     1.42x
    1.0472     -0.008617        -0.008660       -0.010000     1.16x
    1.5708     -0.010000        -0.010000       -0.010000     1.00x
    2.0944     -0.008704        -0.008660       -0.010000     1.15x
  -> the sin(phi) form tracks the exact shift; Eq.(12) overestimates by
     1/sin(phi), which at phi = pi/7 is a factor of 

In [4]:
# =============================================================================
# FINAL CONSISTENCY CHECK
# Produces every number quoted in the letter, all from ONE tier of the
# calculation, so the delta=0 and delta=0.05 figures cannot disagree.
#
# Tier used: sin(gamma/N) factor  +  measured residual-block overlap O_R
#            applied to the eps_j*eps_{j+1} term.  2*delta^2 left bare
#            (Cauchy-Schwarz, no structure factor).
# Requires in scope: w{}, configs{}, epsilon(), complex_overlap()
# =============================================================================
import numpy as np

GAMMA, TARGET = np.pi, 0.31
DOMI, RIDX    = [2, 4, 6], [0, 1, 3, 5, 7, 8]
DELTA_EST     = 0.05

def O_R(pj, pk):
    """Residual-block overlap Tr(rho~_R(j) rho~_R(k)).  The R block is diagonal
       under the SI S10 noise model (same argument that gives rho_DR = 0), so
       this is computable from the measured probability vectors alone."""
    vj = np.array([max(pj[i], 0) for i in RIDX])
    vk = np.array([max(pk[i], 0) for i in RIDX])
    if vj.sum() <= 0 or vk.sum() <= 0:
        return 0.0
    return float((vj/vj.sum()) @ (vk/vk.sum()))

def bound(seq, delta, use_sin=True, use_OR=True):
    N, tot = len(seq) - 1, 0.0
    f = np.sin(GAMMA/N) if use_sin else 1.0
    for i in range(N):
        (ij, aj), (ik, ak) = seq[i], seq[i+1]
        tj = aj if aj is not None else w[ij][0]
        tk = ak if ak is not None else w[ik][0]
        pj, pk = w[ij][1], w[ik][1]
        OD = abs(complex_overlap(tj, pj, tk, pk))**2
        ej, ek = epsilon(pj), epsilon(pk)
        gR = O_R(pj, pk) if use_OR else 1.0
        tot += np.arctan(f*(ej*ek*gR + 2*delta**2)/((1-ej)*(1-ek)*OD))
    return tot

grid = np.linspace(0, 1.0, 10001)
def dstar(seq):
    v = np.array([bound(seq, d) for d in grid])
    return grid[np.argmax(v > TARGET)] if (v > TARGET).any() else np.nan

print("=" * 78)
print("O_R (residual-block overlap) actually measured, per partition")
print("=" * 78)
allOR = []
for lbl, seq in configs.items():
    vals = [O_R(w[seq[i][0]][1], w[seq[i+1][0]][1]) for i in range(len(seq)-1)]
    allOR += vals
    print(f"  {lbl}: mean {np.mean(vals):.3f}   min {min(vals):.3f}   max {max(vals):.3f}")
print(f"  -> overall mean {np.mean(allOR):.3f}   QUOTE THIS as '~{np.mean(allOR):.1f}' in the text")

print("\n" + "=" * 78)
print("ALL LETTER NUMBERS, one tier throughout")
print("=" * 78)
print(f"{'N':>5} {'published':>10} | {'delta=0':>9} {'delta=0.05':>11} {'delta*':>8}")
print(f"{'':>5} {'Table IV':>10} | {'refined':>9} {'refined':>11} {'':>8}")
print("-" * 78)
pub = {'N=3':0.0325,'N=4':0.0314,'N=5':0.0568,'N=6':0.0406,'N=7':0.0472}
b0s, bes, dss = [], [], []
for lbl, seq in configs.items():
    b0, be, ds = bound(seq, 0.0), bound(seq, DELTA_EST), dstar(seq)
    b0s.append(b0); bes.append(be); dss.append(ds)
    print(f"{lbl:>5} {100*pub[lbl]/np.pi:>9.1f}% | {100*b0/np.pi:>8.1f}% "
          f"{100*be/np.pi:>10.1f}% {ds:>8.3f}")

print("-" * 78)
print(f"  published range      : {100*min(pub.values())/np.pi:.1f}% - {100*max(pub.values())/np.pi:.1f}% of pi")
print(f"  refined, delta = 0   : {100*min(b0s)/np.pi:.1f}% - {100*max(b0s)/np.pi:.1f}% of pi")
print(f"  refined, delta = 0.05: {100*min(bes)/np.pi:.1f}% - {100*max(bes)/np.pi:.1f}% of pi"
      f"   (quote the MAX: {100*max(bes)/np.pi:.1f}%)")
print(f"  binding delta*       : {min(dss):.3f}   -> quote {min(dss):.2f}")
print(f"  coherent population  : {100*min(dss)**2:.1f}%")
print(f"  margin over estimate : {min(dss)/DELTA_EST:.1f}x")

print("\n" + "=" * 78)
print("SENTENCE CHECK")
print("=" * 78)
ok = max(bes) <= max(b0s)
print(f"  Is delta=0.05 max ({100*max(bes)/np.pi:.1f}%) <= delta=0 max ({100*max(b0s)/np.pi:.1f}%)? {ok}")
print("   -> if False, the closing clause must read 'remain below the previously")
print("      quoted values' and NOT 'remain at this level'.")

O_R (residual-block overlap) actually measured, per partition
  N=3: mean 0.257   min 0.191   max 0.310
  N=4: mean 0.203   min 0.153   max 0.230
  N=5: mean 0.219   min 0.200   max 0.234
  N=6: mean 0.236   min 0.170   max 0.311
  N=7: mean 0.236   min 0.174   max 0.295
  -> overall mean 0.230   QUOTE THIS as '~0.2' in the text

ALL LETTER NUMBERS, one tier throughout
    N  published |   delta=0  delta=0.05   delta*
        Table IV |   refined     refined         
------------------------------------------------------------------------------
  N=3       1.0% |      0.2%        1.2%    0.162
  N=4       1.0% |      0.1%        0.9%    0.177
  N=5       1.8% |      0.2%        0.9%    0.184
  N=6       1.3% |      0.1%        0.8%    0.191
  N=7       1.5% |      0.1%        0.8%    0.195
------------------------------------------------------------------------------
  published range      : 1.0% - 1.8% of pi
  refined, delta = 0   : 0.1% - 0.2% of pi
  refined, delta = 0.05: 0.8% - 1.

In [5]:
# =============================================================================
# SIGNIFICANCE TEST ON THE HARMONIC COMPONENT
# Is the fitted oscillation amplitude distinguishable from zero?
# Requires in scope: w{}
# =============================================================================
import numpy as np
from scipy import stats

DOMI = [2, 4, 6]
ks   = sorted(w)
th   = np.array([w[k][0] for k in ks])
eps  = np.array([1 - sum(max(w[k][1][i], 0) for i in DOMI) for k in ks])
p2s  = np.array([w[k][1][2] + w[k][1][6] for k in ks])
p11  = np.array([w[k][1][4] for k in ks])

# residual after removing the incoherent (population-driven) dependence
A       = np.vstack([p2s, p11]).T
c, *_   = np.linalg.lstsq(A, eps, rcond=None)
r       = eps - A @ c

# harmonic fit
X       = np.vstack([np.ones_like(th), np.cos(th), np.sin(th)]).T
cc, *_  = np.linalg.lstsq(X, r, rcond=None)
n, p    = len(th), X.shape[1]
dof     = n - p
rss_f   = np.sum((r - X @ cc)**2)
Cov     = (rss_f/dof) * np.linalg.inv(X.T @ X)
a, b    = cc[1], cc[2]
sa, sb  = np.sqrt(Cov[1, 1]), np.sqrt(Cov[2, 2])
B       = np.hypot(a, b)
sB      = np.sqrt((a*sa)**2 + (b*sb)**2)/B if B > 0 else np.nan

# F-test: harmonic model vs constant
X0      = np.ones((n, 1))
c0, *_  = np.linalg.lstsq(X0, r, rcond=None)
rss_0   = np.sum((r - X0 @ c0)**2)
F       = ((rss_0 - rss_f)/2) / (rss_f/dof)
pval    = 1 - stats.f.cdf(F, 2, dof)

print("=" * 66)
print(f"  n = {n} angles,  dof = {dof}")
print(f"  cos coeff a = {a:+.5f} +- {sa:.5f}   t = {a/sa:+.2f}")
print(f"  sin coeff b = {b:+.5f} +- {sb:.5f}   t = {b/sb:+.2f}")
print(f"  amplitude B = {B:.5f} +- {sB:.5f}   ({B/sa:.2f} sigma from zero)")
print("-" * 66)
print(f"  F-test vs constant:  F(2,{dof}) = {F:.3f}   p = {pval:.3f}")
print(f"  |t| threshold for 5% significance at dof={dof}: {stats.t.ppf(0.975, dof):.2f}")
print("-" * 66)
print(f"  VERDICT: {'no evidence of an oscillatory contribution' if pval > 0.05 else 'OSCILLATION DETECTED - revisit'}")
print(f"  Sensitivity floor (2-sigma on a coeff): {2*sa:.4f}")
print(f"  -> oscillations below this amplitude are undetectable with this data")
print("=" * 66)
print(f"\n  QUOTE IN LETTER: amplitude {B:.3f} +- {max(sa,sb):.3f}, F-test p = {pval:.2f}")
print(f"  delta = sqrt(B) = {np.sqrt(B):.3f}")

  n = 17 angles,  dof = 14
  cos coeff a = +0.00136 +- 0.01137   t = +0.12
  sin coeff b = +0.00227 +- 0.00921   t = +0.25
  amplitude B = 0.00265 +- 0.00983   (0.23 sigma from zero)
------------------------------------------------------------------
  F-test vs constant:  F(2,14) = 0.039   p = 0.962
  |t| threshold for 5% significance at dof=14: 2.14
------------------------------------------------------------------
  VERDICT: no evidence of an oscillatory contribution
  Sensitivity floor (2-sigma on a coeff): 0.0227
  -> oscillations below this amplitude are undetectable with this data

  QUOTE IN LETTER: amplitude 0.003 +- 0.011, F-test p = 0.96
  delta = sqrt(B) = 0.051


In [11]:
# =============================================================================
# SIGNIFICANCE TEST ON THE HARMONIC COMPONENT
# Is the fitted oscillation amplitude distinguishable from zero?
# Tests BOTH the fundamental and the second harmonic: the RZ gates sit between
# two sqrt(X01) pulses, so population (not just phase) depends on theta and
# 2-theta structure is possible in principle.
# Requires in scope: w{}
# =============================================================================
import numpy as np
from scipy import stats

DOMI = [2, 4, 6]
ks   = sorted(w)
th   = np.array([w[k][0] for k in ks])
eps  = np.array([1 - sum(max(w[k][1][i], 0) for i in DOMI) for k in ks])
p2s  = np.array([w[k][1][2] + w[k][1][6] for k in ks])
p11  = np.array([w[k][1][4] for k in ks])

# residual after removing the incoherent (population-driven) dependence
A     = np.vstack([p2s, p11]).T
c, *_ = np.linalg.lstsq(A, eps, rcond=None)
r     = eps - A @ c
n     = len(th)


def harmonic_fit(r, th, orders):
    """Fit constant + cos/sin terms at the given harmonic orders."""
    cols = [np.ones_like(th)]
    for m in orders:
        cols += [np.cos(m * th), np.sin(m * th)]
    X      = np.vstack(cols).T
    cc, *_ = np.linalg.lstsq(X, r, rcond=None)
    dof    = len(th) - X.shape[1]
    rss    = np.sum((r - X @ cc) ** 2)
    Cov    = (rss / dof) * np.linalg.inv(X.T @ X)
    out    = {}
    for i, m in enumerate(orders):
        a  = cc[1 + 2 * i]
        b  = cc[2 + 2 * i]
        sa = np.sqrt(Cov[1 + 2 * i, 1 + 2 * i])
        sb = np.sqrt(Cov[2 + 2 * i, 2 + 2 * i])
        B  = np.hypot(a, b)
        sB = np.sqrt((a * sa) ** 2 + (b * sb) ** 2) / B if B > 0 else np.nan
        out[m] = dict(a=a, b=b, sa=sa, sb=sb, B=B, sB=sB)
    return out, rss, dof


# constant-only null model
rss_0 = np.sum((r - r.mean()) ** 2)

# full model: fundamental + second harmonic
res, rss_f, dof_f = harmonic_fit(r, th, [1, 2])
F_all = ((rss_0 - rss_f) / 4) / (rss_f / dof_f)
p_all = 1 - stats.f.cdf(F_all, 4, dof_f)

# fundamental only, for comparison with the currently quoted number
res1, rss_1, dof_1 = harmonic_fit(r, th, [1])
F_1 = ((rss_0 - rss_1) / 2) / (rss_1 / dof_1)
p_1 = 1 - stats.f.cdf(F_1, 2, dof_1)

# does the second harmonic add anything over the fundamental alone?
F_2 = ((rss_1 - rss_f) / 2) / (rss_f / dof_f)
p_2 = 1 - stats.f.cdf(F_2, 2, dof_f)

# ---------------------------------------------------------------- reporting
B_tot = np.hypot(res[1]['B'], res[2]['B'])
ok    = (p_all > 0.05) and (p_2 > 0.05)
floor = 2 * max(res[1]['sa'], res[1]['sb'], res[2]['sa'], res[2]['sb'])

if ok:
    verdict = "no evidence of an oscillatory contribution at either order"
else:
    verdict = "OSCILLATION DETECTED - revisit"

amp1 = res1[1]['B']
err1 = max(res1[1]['sa'], res1[1]['sb'])

print("=" * 70)
print(f"  n = {n} angles")
print("-" * 70)

for m in (1, 2):
    d   = res[m]
    lbl = "fundamental" if m == 1 else "2nd harmonic"
    sig = d['B'] / max(d['sa'], d['sb'])
    print(f"  {lbl:>13}:  cos {d['a']:+.5f} +- {d['sa']:.5f}"
          f"   sin {d['b']:+.5f} +- {d['sb']:.5f}")
    print(f"  {'':>13}   amplitude B = {d['B']:.5f}   ({sig:.2f} sigma from zero)")

print("-" * 70)
print(f"  fundamental only vs constant:       F(2,{dof_1}) = {F_1:.3f}   p = {p_1:.3f}")
print(f"  both harmonics vs constant:         F(4,{dof_f}) = {F_all:.3f}   p = {p_all:.3f}")
print(f"  2nd harmonic added to fundamental:  F(2,{dof_f}) = {F_2:.3f}   p = {p_2:.3f}")
print("-" * 70)
print(f"  VERDICT: {verdict}")
print(f"  Sensitivity floor (2-sigma on a coefficient): {floor:.4f}")
print("  -> oscillations below this amplitude are undetectable with this data")
print("=" * 70)
print()
print("  FUNDAMENTAL ONLY (as currently quoted in the letter):")
print(f"    amplitude {amp1:.3f} +- {err1:.3f},  p = {p_1:.2f},"
      f"  delta = {np.sqrt(amp1):.3f}")
print()
print("  BOTH HARMONICS (conservative):")
print(f"    total amplitude {B_tot:.3f},  p = {p_all:.2f},"
      f"  delta = {np.sqrt(B_tot):.3f}")
print("    -> compare against delta* = 0.16")

  n = 17 angles
----------------------------------------------------------------------
    fundamental:  cos -0.00094 +- 0.01257   sin +0.00235 +- 0.00973
                  amplitude B = 0.00253   (0.20 sigma from zero)
   2nd harmonic:  cos -0.00624 +- 0.01161   sin +0.00589 +- 0.01048
                  amplitude B = 0.00858   (0.74 sigma from zero)
----------------------------------------------------------------------
  fundamental only vs constant:       F(2,14) = 0.039   p = 0.962
  both harmonics vs constant:         F(4,12) = 0.175   p = 0.947
  2nd harmonic added to fundamental:  F(2,12) = 0.315   p = 0.736
----------------------------------------------------------------------
  VERDICT: no evidence of an oscillatory contribution at either order
  Sensitivity floor (2-sigma on a coefficient): 0.0251
  -> oscillations below this amplitude are undetectable with this data

  FUNDAMENTAL ONLY (as currently quoted in the letter):
    amplitude 0.003 +- 0.011,  p = 0.96,  delta = 0.05